# Week 8: Advanced optimisation, the genetic algorithm
ENGI 1151 Computational Tools for Engineers and Scientists

Run each cell with **Shift + Enter**, from top to bottom: later cells use variables from earlier ones. The full explanations are on the week 8 notes page.

First install PyGAD (once per session in this browser notebook; on your own computer `pip install pygad` once is enough).

In [ ]:
%pip install -q pygad


## The mountain range
Four peaks; the highest, 1250 m, is at (6.5, 6.0) km. `h(x, y)` is the height and `grad_h(x, y)` its gradient.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# the mountain range: four peaks (cx, cy, height above base a, width w), in km and m
PEAKS = [(6.5, 6.0, 1100, 1.3), (2.5, 7.5, 850, 0.9), (3.0, 2.5, 750, 1.0), (8.2, 2.0, 650, 0.8)]

def h(x, y):
    """Height of the mountain range (m) at x, y (km); works with arrays."""
    z = 150 + 0 * x
    for cx, cy, a, w in PEAKS:
        z = z + a * np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * w**2))
    return z

def grad_h(x, y):
    """Gradient of h: (dh/dx, dh/dy)."""
    gx, gy = 0 * x, 0 * y
    for cx, cy, a, w in PEAKS:
        e = a * np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * w**2)) / w**2
        gx, gy = gx - e * (x - cx), gy - e * (y - cy)
    return gx, gy


## 1. Gradient ascent gets trapped on local peaks
1000 hikers climb from random starting points.

In [ ]:
def climb(x, y, alpha=0.001, n_max=300):
    """Gradient ascent; x and y can be arrays of many starting points."""
    for k in range(n_max):
        gx, gy = grad_h(x, y)
        x = np.clip(x + alpha * gx, 0, 10)
        y = np.clip(y + alpha * gy, 0, 10)
    return x, y

rng = np.random.default_rng(1)
x0, y0 = rng.uniform(0, 10, (2, 1000))      # 1000 random starting points
x, y = climb(x0, y0)                        # all hikers climb at once

tops = np.array([h(cx, cy) for cx, cy, a, w in PEAKS])
ends = h(x, y)
for top in tops:
    print("peak of %4.0f m: %3d hikers" % (top, np.sum(abs(ends - top) < 5)))
print("still on a slope:", np.sum(np.min(abs(ends[:, None] - tops), axis=1) >= 5))


## 4. A random population and its fitness

In [ ]:
rng = np.random.default_rng(3)
pop_size, n_genes = 12, 2
population = rng.uniform(0, 10, (pop_size, n_genes))   # generation 0

fitness = h(population[:, 0], population[:, 1])       # height of every chromosome
order = np.argsort(fitness)[::-1]                      # rank: best first
print("rank   x (km)   y (km)   h (m)")
for rank, i in enumerate(order, start=1):
    print("%3d   %6.2f   %6.2f   %6.0f" % (rank, *population[i], fitness[i]))

X, Y = np.meshgrid(np.linspace(0, 10, 200), np.linspace(0, 10, 200))
plt.contourf(X, Y, h(X, Y), levels=16, cmap="Purples")
plt.plot(population[:, 0], population[:, 1], "o", color="#B3261E", mec="white", ms=9)
plt.xlabel("x (km)"); plt.ylabel("y (km)"); plt.gca().set_aspect("equal")
plt.title("Generation 0: 12 random chromosomes")
plt.show()


## 5. Single-point crossover

In [ ]:
import numpy as np

def single_point_crossover(a, b, rng):
    """Cut both parents at the same random point and swap the tails."""
    cut = rng.integers(1, len(a))            # between gene cut and gene cut + 1
    child1 = np.concatenate([a[:cut], b[cut:]])
    child2 = np.concatenate([b[:cut], a[cut:]])
    return child1, child2, cut

rng = np.random.default_rng(4)
A = np.array([2.4, 7.1, 0.8, 5.5, 3.3, 9.0])
B = np.array([6.6, 1.9, 4.2, 8.7, 0.5, 2.8])
c1, c2, cut = single_point_crossover(A, B, rng)
print("cut after gene", cut)
print("child 1:", c1)
print("child 2:", c2)


## 6. Selection and mutation

In [ ]:
rng = np.random.default_rng(3)
population = rng.uniform(0, 10, (12, 2))
fitness = h(population[:, 0], population[:, 1])

def tournament(population, fitness, n_parents, k, rng):
    """Pick k chromosomes at random, keep the fittest; repeat n_parents times."""
    parents = []
    for _ in range(n_parents):
        entrants = rng.choice(len(population), size=k, replace=False)
        winner = entrants[np.argmax(fitness[entrants])]
        parents.append(population[winner])
    return np.array(parents)

def roulette(population, fitness, n_parents, rng):
    """Choose parents with probability proportional to their fitness."""
    p = fitness / fitness.sum()
    return population[rng.choice(len(population), size=n_parents, p=p)]

print("population mean height: %.0f m" % fitness.mean())
for name, parents in [("tournament k=3", tournament(population, fitness, 6, 3, rng)),
                      ("roulette wheel", roulette(population, fitness, 6, rng))]:
    print("%-15s parents' mean height: %.0f m" % (name, h(parents[:, 0], parents[:, 1]).mean()))


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
children = np.array([[6.3, 5.1], [2.4, 7.1], [8.0, 6.0], [6.4, 2.6]])

p_mutation, step = 0.3, 1.0
mutate = rng.random(children.shape) < p_mutation        # which genes mutate
noise = rng.normal(0, step, children.shape)            # random perturbation
mutated = np.clip(children + mutate * noise, 0, 10)    # stay inside the map
print(mutate)
print(np.round(mutated, 2))


## 8. A complete genetic algorithm in NumPy

In [ ]:
def genetic_algorithm(pop_size=20, n_parents=10, n_elite=2, generations=40,
                      p_mutation=0.3, step=1.0, seed=None, verbose=False):
    rng = np.random.default_rng(seed)
    pop = rng.uniform(0, 10, (pop_size, 2))                 # 1. random population
    for gen in range(generations):
        fit = h(pop[:, 0], pop[:, 1])                       # 2. fitness
        order = np.argsort(fit)[::-1]
        pop, fit = pop[order], fit[order]                   #    rank, best first
        if verbose and gen % 5 == 0:
            print("generation %2d: best (%.3f, %.3f), h = %6.1f m, mean %6.1f m"
                  % (gen, pop[0, 0], pop[0, 1], fit[0], fit.mean()))
        parents = pop[:n_parents]                           # 4. selection: top-ranked
        children = []
        while len(children) < pop_size - n_elite:
            i, j = rng.choice(n_parents, size=2, replace=False)
            a, b = parents[i], parents[j]
            children += [[a[0], b[1]], [b[0], a[1]]]        # 5. crossover: swap y
        children = np.array(children[:pop_size - n_elite])
        mutate = rng.random(children.shape) < p_mutation    # 6. mutation
        children = np.clip(children + mutate * rng.normal(0, step, children.shape), 0, 10)
        pop = np.vstack([pop[:n_elite], children])          # 7. elitism + children
    fit = h(pop[:, 0], pop[:, 1])
    best = np.argmax(fit)
    return pop[best], fit[best]

x_best, h_best = genetic_algorithm(seed=4, verbose=True)
print("best chromosome:", np.round(x_best, 3), "height %.1f m" % h_best)
print("evaluations of h:", 20 * 41)


## 9. Ten runs, ten answers, and polishing with a gradient method

In [ ]:
def genetic_algorithm(pop_size=20, n_parents=10, n_elite=2, generations=40,
                      p_mutation=0.3, step=1.0, seed=None):
    rng = np.random.default_rng(seed)
    pop = rng.uniform(0, 10, (pop_size, 2))
    for gen in range(generations):
        fit = h(pop[:, 0], pop[:, 1])
        pop = pop[np.argsort(fit)[::-1]]
        parents = pop[:n_parents]
        children = []
        while len(children) < pop_size - n_elite:
            i, j = rng.choice(n_parents, size=2, replace=False)
            a, b = parents[i], parents[j]
            children += [[a[0], b[1]], [b[0], a[1]]]
        children = np.array(children[:pop_size - n_elite])
        mutate = rng.random(children.shape) < p_mutation
        children = np.clip(children + mutate * rng.normal(0, step, children.shape), 0, 10)
        pop = np.vstack([pop[:n_elite], children])
    fit = h(pop[:, 0], pop[:, 1])
    return pop[np.argmax(fit)], fit.max()

print("run     x (km)    y (km)    h (m)")
for run in range(10):
    best, height = genetic_algorithm(seed=run)
    print("%3d   %7.3f   %7.3f   %7.1f" % (run + 1, best[0], best[1], height))


In [ ]:
from scipy.optimize import minimize

ga_best = np.array([6.477, 6.013])                 # the answer of run 1 above
res = minimize(lambda p: -h(p[0], p[1]), ga_best)  # maximise h = minimise -h
print("GA answer:       (%.3f, %.3f), h = %.2f m" % (*ga_best, h(*ga_best)))
print("after polishing: (%.4f, %.4f), h = %.2f m" % (*res.x, -res.fun))


## 10. PyGAD: the handout example
PyGAD **maximises** the fitness: return $-f$ to minimise $f$.

In [ ]:
import numpy as np
import pygad

def fitness(ga_instance, solution, solution_idx):
    x, y = solution
    f = ((x - np.pi)**2 + (y - np.pi/2)**2 + np.sin(3*x + np.pi/3)
         + np.sin(4*y - np.pi/4) * np.exp(-((x - 3)**2 + (y - 3)**2)))
    return -f                       # PyGAD maximises: maximise -f to minimise f

ga_instance = pygad.GA(num_generations=200, num_parents_mating=10,
                       sol_per_pop=12, num_genes=2, fitness_func=fitness,
                       init_range_low=-10, init_range_high=10,
                       parent_selection_type="rank", keep_elitism=1,
                       crossover_type="single_point", mutation_type="random",
                       mutation_percent_genes=50, random_seed=1,
                       suppress_warnings=True)
ga_instance.run()
solution, solution_fitness, solution_idx = ga_instance.best_solution()
print("best (x, y) =", np.round(solution, 4))
print("minimum f   =", round(-solution_fitness, 4))
print("found in generation", ga_instance.best_solution_generation)
ga_instance.plot_fitness(title="Best fitness (= -f) per generation")


Ten runs without a random seed:

In [ ]:
import numpy as np
import pygad

def fitness(ga_instance, solution, solution_idx):
    x, y = solution
    f = ((x - np.pi)**2 + (y - np.pi/2)**2 + np.sin(3*x + np.pi/3)
         + np.sin(4*y - np.pi/4) * np.exp(-((x - 3)**2 + (y - 3)**2)))
    return -f

print("run       x        y        f(x, y)")
for run in range(1, 11):
    ga_instance = pygad.GA(num_generations=100, num_parents_mating=10,
                           sol_per_pop=12, num_genes=2, fitness_func=fitness,
                           init_range_low=-10, init_range_high=10,
                           parent_selection_type="rank", keep_elitism=1,
                           mutation_percent_genes=50, suppress_warnings=True)
    ga_instance.run()
    s, fit, _ = ga_instance.best_solution()
    print("%3d  %8.4f %8.4f %10.5f" % (run, s[0], s[1], -fit))


## 11. A beam from a catalogue of standard sizes

In [ ]:
import numpy as np
import pygad

L, w, E, sigma_allow = 6000.0, 20.0, 200000.0, 165.0   # mm, N/mm (= kN/m), MPa, MPa
widths = list(range(40, 205, 5))                       # mm, standard sizes
depths = list(range(100, 610, 10))                     # mm

def beam(b, d):
    """Mass (kg/m), bending stress (MPa) and mid-span deflection (mm)."""
    M = w * L**2 / 8
    sigma = 6 * M / (b * d**2)
    delta = 5 * w * L**4 / (384 * E * b * d**3 / 12)
    return 7850 * b * d * 1e-6, sigma, delta

def fitness(ga_instance, solution, solution_idx):
    b, d = solution
    mass, sigma, delta = beam(b, d)
    penalty = (max(0, sigma / sigma_allow - 1) + max(0, delta / (L / 360) - 1)
               + max(0, d / (6 * b) - 1)) * 1000
    return -(mass + penalty)                           # minimise the mass

for run in range(5):
    ga_instance = pygad.GA(num_generations=100, num_parents_mating=10,
                           sol_per_pop=30, num_genes=2, fitness_func=fitness,
                           gene_space=[widths, depths], gene_type=int,
                           parent_selection_type="tournament", keep_elitism=2,
                           mutation_percent_genes=50, random_seed=run,
                           suppress_warnings=True)
    ga_instance.run()
    (b, d), fit, _ = ga_instance.best_solution()
    mass, sigma, delta = beam(b, d)
    print("run %d: b = %3d mm, d = %3d mm, mass %.1f kg/m, stress %.0f MPa, deflection %.1f mm"
          % (run + 1, b, d, mass, sigma, delta))


## Exercises
1. Write `two_point_crossover(a, b, rng)` and `uniform_crossover(a, b, rng)` and apply them to the parents of section 5.
2. Run `genetic_algorithm` 100 times for population sizes 10, 20 and 40 and count how often the best height exceeds 1245 m. Compare with gradient ascent from a single random start.
3. Repeat the ten PyGAD runs with `parent_selection_type="tournament"`, then `"rws"`, then `keep_elitism=0`. Which setting is the most reliable?
4. Add a third gene to the beam, the steel grade: 0 for S275 (allowable stress 165 MPa, cost factor 1.0), 1 for S355 (213 MPa, cost factor 1.1). Does the higher grade pay off?
5. Minimise the handout's function with `scipy.optimize.minimize` from (2, 2), (3, 3) and (3, 1.5). Compare with the GA.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3


In [ ]:
# Exercise 4


In [ ]:
# Exercise 5
